# 02 - Dashboard Data Checks

Use this notebook to inspect the timestamped update snapshots that feed the dashboard and future forecasting pipeline.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
from entsoe_realtime.config import load_settings
from entsoe_realtime.storage import collect_snapshot_summary

settings = load_settings()
summary = collect_snapshot_summary(settings.update_dir, settings.update_manifest)
summary

## Load One Series

In [ ]:
country = 'BE'
variable = 'actual_load'
files = sorted((settings.update_dir / country / variable).glob('*/*/*.csv'))
files[:3], files[-3:]

In [ ]:
if files:
    frame = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
    frame['timestamp_utc'] = pd.to_datetime(frame['timestamp_utc'], utc=True)
    display(frame.tail())
    frame.groupby('source')['value'].describe()
else:
    print('No update snapshots yet. Run notebooks/01_backfill_entsoe_data.ipynb or scripts/fetch_once.py first.')

In [ ]:
if files:
    frame.set_index('timestamp_utc').groupby('source')['value'].tail(500).plot(figsize=(14, 5), title=f'{country} {variable}')